# MOT-MIC Example

This notebook demonstrates the full MOT-MIC workflow on a small synthetic dataset that mimics paired primary tumor cells and multiple metastatic sites.

For real analyses, replace the simulated matrices with tumor-cell matrices from GSE173958, GSE249057, GSE178318, or GSE277783 after QC and malignant-cell filtering.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(repo_root))

from motmic import MOTMIC, evaluate_binary_labels, simulate_metastasis_dataset
from motmic.interpret import rank_genes_with_shap

primary, metastases, truth = simulate_metastasis_dataset(random_state=7)
primary.shape, {k: v.shape for k, v in metastases.items()}, truth.head()

((240, 100),
 {'lung': (90, 100), 'liver': (90, 100), 'bone': (90, 100)},
          true_MIC true_site aggressive_clone
 cell_id                                     
 P000        False      none   non_aggressive
 P001        False      none   non_aggressive
 P002        False      none   non_aggressive
 P003        False      none   non_aggressive
 P004        False      none   non_aggressive)

## Fit MOT-MIC

The estimator learns a shared latent space, maps primary cells to each metastatic site with unbalanced optimal transport, applies top-k origin filtering, and returns organ-specific MIC scores.

In [2]:
model = MOTMIC(n_components=15, epsilon=0.08, rho=1.2, top_k=1)
result = model.fit_predict(primary, metastases)
scores = result.to_frame()
scores.head()

,lung,liver,bone,pan_MIC_score,organ_specificity,predicted_site
P000,0.0000,0.0,0.0,0.0000,0.0,lung
P001,0.0000,0.0,0.0,0.0000,0.0,lung
P002,0.0000,0.0,0.0,0.0000,0.0,lung
P003,0.0000,0.0,0.0,0.0000,0.0,lung
P004,0.0774,0.0,0.0,0.0774,1.0,lung


## Validate Against Lineage-Like Labels

In real GSE173958 analysis, replace `truth['true_MIC']` with aggressive clone labels or clone dissemination labels.

In [3]:
metrics = evaluate_binary_labels(result.pan_score, truth['true_MIC'])
metrics

{'mean_positive_score': 0.685058177239611,
 'mean_negative_score': 0.02546814202225609,
 'AUROC': 0.9867021276595744,
 'AUPRC': 0.9657229824371112}

## Rank Candidate Genes

SHAP is used when available. If SHAP is unavailable, the function falls back to permutation importance so the workflow still runs.

In [4]:
high_mic = result.pan_score >= result.pan_score.quantile(0.8)
ranking = rank_genes_with_shap(primary, high_mic.astype(int))
ranking.head(20)

,gene,importance,method,rank
0,CD44,0.075982,shap,1
1,VIM,0.056695,shap,2
2,RHOD,0.052375,shap,3
3,FN1,0.049428,shap,4
4,TACSTD2,0.046435,shap,5
5,S100A14,0.042618,shap,6
6,AXL,0.039587,shap,7
7,L1CAM,0.011829,shap,8
8,ITGA5,0.010555,shap,9
9,ITGB1,0.008588,shap,10


## Save Results

The same tables are produced by `scripts/run_example.py`.

In [5]:
from pathlib import Path
Path('../results').mkdir(exist_ok=True)
scores.to_csv('../results/notebook_motmic_scores.csv')
ranking.to_csv('../results/notebook_gene_ranking.csv', index=False)